# Gourmand Rising: what sells in fragrance and what comes next in 2027

**Business question:** Which gourmand scent terms (vanilla, chocolate, caramel) are growing in the Netherlands, and where should a fragrance brand or supplier place its next bets?

*Personal project, public data (Google Trends). Search interest is a relative index (0 to 100), not sales.*

**How to use this notebook**
1. Put your two Google Trends CSV files in a folder called `data` next to this notebook.
2. Check the file names in the next cell match yours.
3. Run the cells from the top (Shift + Enter). Each cell has a short note on what it does.

In [ ]:
import re, io
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from statsmodels.tsa.exponential_smoothing.ets import ETSModel

plt.rcParams.update({
    "figure.figsize": (11, 4.5),
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25,
})

DATA_DIR = Path("data")
OUT_DIR = Path("output"); OUT_DIR.mkdir(exist_ok=True)

# Change these names if your files are called something else.
FILES = {
    "Dutch terms":   DATA_DIR / "trends_NL_dutch_terms_5y.csv",
    "English terms": DATA_DIR / "trends_NL_english_terms_5y.csv",
}
FILES = {k: v for k, v in FILES.items() if v.exists()}   # use whichever files you have
print("Files found:", list(FILES))

## 1. Load and clean
Google Trends CSV files start with a few header lines, and use `<1` for very small values. This cell skips the header, turns `<1` into 0.5, and tidies the column names.

In [ ]:
def load_trends(path):
    lines = Path(path).read_text(encoding="utf-8-sig").splitlines()
    start = next(i for i, l in enumerate(lines) if re.match(r'^"?(Week|Day|Month|Time)"?\s*,', l))
    df = pd.read_csv(io.StringIO("\n".join(lines[start:])))
    date_col = df.columns[0]
    df[date_col] = pd.to_datetime(df[date_col])
    df = df.set_index(date_col).sort_index()
    df.index.name = "date"
    df.columns = [re.sub(r"\s*:\s*\(.*\)$", "", c).strip() for c in df.columns]
    df = df.replace("<1", 0.5).apply(pd.to_numeric, errors="coerce")
    return df

def is_monthly(idx):
    # Newer Google Trends downloads are monthly, older ones are weekly.
    return len(idx) > 2 and pd.Series(idx).diff().median() > pd.Timedelta(days=20)

def drop_partial(df):
    # Drop the most recent period if it is still running.
    if is_monthly(df.index):
        if df.index[-1] + pd.offsets.MonthBegin(1) > pd.Timestamp.today():
            return df.iloc[:-1]
        return df
    return df.iloc[:-1]

data = {name: load_trends(p) for name, p in FILES.items()}
data = {name: drop_partial(df) for name, df in data.items()}

# Terms you typed by mistake (for example "parfum." with a dot) can be dropped here.
DROP_TERMS = ["parfum."]
data = {name: df.drop(columns=DROP_TERMS, errors="ignore") for name, df in data.items()}

for name, df in data.items():
    unit = "months" if is_monthly(df.index) else "weeks"
    print(f"{name}: {df.shape[0]} {unit}, {df.index.min().date()} to {df.index.max().date()}")
    low = (df <= 0.5).mean().round(2).rename(f"share of {unit} with ~zero interest")
    print(low.to_string(), "\n")

**Data quality note:** terms with many near-zero weeks are low volume. Treat their week-to-week moves as noisy and look at monthly totals instead.

## 2. First look: weekly interest over time

In [ ]:
for name, df in data.items():
    monthly_data = is_monthly(df.index)
    win, unit = (3, "month") if monthly_data else (4, "week")
    fig, ax = plt.subplots()
    for col in df.columns:
        ax.plot(df.index, df[col].rolling(win, min_periods=1).mean(), label=col, linewidth=2)
    ax.set_title(f"{name}: search interest in the Netherlands ({win}-{unit} average)")
    ax.set_ylabel("Relative interest (0-100)")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"01_{name.lower().replace(' ', '_')}_over_time.png", dpi=150)
    plt.show()

**Note:** each CSV is scaled on its own, so 100 means "the highest point among the terms in that file". Compare terms **within** a file, never across the two files.

## 3. Which term dominates? (share of interest within each file)

In [ ]:
for name, df in data.items():
    share = (df.sum() / df.sum().sum()).sort_values(ascending=False)
    print(name)
    print((share * 100).round(1).astype(str).add("%").to_string(), "\n")

## 4. Seasonality: when does interest peak each year?

In [ ]:
for name, df in data.items():
    by_month = df.groupby(df.index.month).mean()
    fig, ax = plt.subplots()
    for col in by_month.columns:
        ax.plot(by_month.index, by_month[col], marker="o", label=col)
    ax.set_xticks(range(1, 13))
    ax.set_xlabel("Month of year")
    ax.set_title(f"{name}: average interest by month")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / f"02_{name.lower().replace(' ', '_')}_seasonality.png", dpi=150)
    plt.show()

    holiday = df[df.index.month.isin([11, 12])].mean()
    rest = df[~df.index.month.isin([11, 12])].mean()
    ratio = (holiday / rest.replace(0, np.nan)).round(2).rename("Nov-Dec vs rest of year (x)")
    print(ratio.to_string(), "\n")

### Is the yearly Nov-Dec peak getting bigger?

In [ ]:
def season_peaks(s):
    out = {}
    for y in range(s.index.year.min(), s.index.year.max() + 1):
        w = s[(s.index >= f"{y}-10-01") & (s.index < f"{y+1}-02-01")]
        monthly_s = is_monthly(s.index)
        complete = s.index.max() >= pd.Timestamp(f"{y+1}-01-01" if monthly_s else f"{y+1}-01-25")
        if len(w) >= (3 if monthly_s else 8):
            out[f"{y}/{str(y+1)[2:]}" + ("" if complete else " (season still running)")] = w.max()
    return pd.Series(out, name="peak interest, Oct-Jan")

name = list(data)[0]                      # first file = Dutch terms
top = data[name].mean().idxmax()          # most searched term
print(f"Most searched term in '{name}': {top}")
print(season_peaks(data[name][top]).to_string())

## 5. The September 2026 spike: was it one term or all of them?

In [ ]:
def find_spikes(df, k=2.5, min_terms=2):
    z = (df - df.mean()) / df.std(ddof=0)
    flagged = z.gt(k)
    hits = flagged.sum(axis=1)
    return df[hits >= min_terms].assign(terms_spiking=hits[hits >= min_terms])

for name, df in data.items():
    sp = find_spikes(df)
    print(f"{name}: periods where {2}+ terms spiked together")
    print(sp.to_string() if len(sp) else "none", "\n")

**Caution:** a spike that hits every term at once from a low base can come from a small group of enthusiasts, a launch, or a viral post. Check *Related queries* in Google Trends before calling it a trend. Park the cause as an open question until you can show it.

## 6. Forecast to the end of 2027 (monthly)

In [ ]:
name = list(data)[0]
df = data[name]
top = df.mean().idxmax()

def to_monthly(series):
    m = series.resample("MS").mean()
    counts = series.resample("MS").count()
    m = m[counts >= (1 if is_monthly(series.index) else 3)]   # drop partial months at the ends
    m.index.freq = "MS"
    return m

# The Sept 2026 spike hit every term at once. By default it is treated as a one-off,
# so the model does not learn a fake yearly September peak. Set SPIKE_WINDOW = None to keep it.
# Monthly files need the whole of Aug and Sep (the spike started in late August).
SPIKE_WINDOW = ("2026-08-01", "2026-09-30") if is_monthly(df.index) else ("2026-08-25", "2026-09-20")

raw = df[top]
cleaned = raw.copy()
if SPIKE_WINDOW:
    cleaned.loc[SPIKE_WINDOW[0]:SPIKE_WINDOW[1]] = np.nan
    cleaned = cleaned.interpolate()

monthly = to_monthly(raw)                 # what actually happened
monthly_fit = to_monthly(cleaned)         # what the forecast learns from
print(f"Forecasting: {top}  ({len(monthly)} months of data)")

def fit_ets(y):
    return ETSModel(y, error="add", trend="add", damped_trend=True,
                    seasonal="add", seasonal_periods=12).fit(disp=False)

# Backtest: hide the last 6 months, forecast them, compare with a simple seasonal guess.
h = 6
test = monthly.iloc[-h:]
bt = fit_ets(monthly_fit.iloc[:-h]).forecast(h)
seasonal_naive = monthly.shift(12).iloc[-h:]
mae_model = float(np.abs(bt.values - test.values).mean())
mae_naive = float(np.nanmean(np.abs(seasonal_naive.values - test.values)))
print(f"Backtest error (MAE, last {h} months): model {mae_model:.1f} vs 'same as last year' {mae_naive:.1f}")
print("Note: the test months include the September 2026 spike, which no model can predict.")

In [ ]:
final = fit_ets(monthly_fit)
steps = 14
pred = final.get_prediction(start=len(monthly_fit), end=len(monthly) + steps - 1).summary_frame(alpha=0.2)
future_index = pd.date_range(monthly_fit.index[-1] + pd.offsets.MonthBegin(1), periods=steps, freq="MS")
pred.index = future_index

fig, ax = plt.subplots()
ax.plot(monthly.index, monthly, label="Actual (monthly average)", linewidth=2)
ax.plot(pred.index, pred["mean"], label="Forecast", linewidth=2, linestyle="--")
ax.fill_between(pred.index, pred["pi_lower"], pred["pi_upper"], alpha=0.2, label="80% range")
ax.set_title(f"'{top}': monthly search interest and forecast")
ax.set_ylabel("Relative interest (0-100)")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(OUT_DIR / "03_forecast.png", dpi=150)
plt.show()

print(pred[["mean", "pi_lower", "pi_upper"]].round(1).to_string())

**How to read the forecast:** it extends the yearly pattern and recent level. The shaded band is statistical uncertainty only. It cannot know about launches, viral moments or new rules. If the backtest error is not clearly better than "same as last year", say so in your write-up.

## 7. SQL: the same numbers using window functions
DuckDB runs real SQL on a table inside Python. Here SQL builds a 3-month rolling average and year-over-year change for every term, and a yearly summary. These are common analyst queries.

In [ ]:
%pip install -q duckdb
import duckdb

def family_of(term):
    t = term.lower()
    if "vanil" in t: return "Vanilla"
    if "chocol" in t: return "Chocolate"
    if "caramel" in t: return "Caramel"
    if "gourmand" in t: return "Gourmand (general)"
    return "Other"

parts = []
for fname, d in data.items():
    m = d.resample("MS").mean()
    full = d.resample("MS").count() >= (1 if is_monthly(d.index) else 3)   # keep only complete months
    m = m.where(full)
    tidy = (m.reset_index()
              .melt(id_vars="date", var_name="term", value_name="interest")
              .dropna(subset=["interest"]))
    tidy["file"] = fname
    tidy["family"] = tidy["term"].map(family_of)
    parts.append(tidy)

monthly_all = pd.concat(parts, ignore_index=True)
monthly_all["type"] = "Actual"
monthly_all["language"] = np.where(monthly_all["file"].str.contains("Dutch"), "Dutch", "English")
print(monthly_all.shape, "rows in the tidy monthly table")
monthly_all.head()

In [ ]:
con = duckdb.connect()
con.register("monthly_all", monthly_all)

monthly_sql = con.sql('''
    SELECT
        date, file, term,
        ROUND(interest, 1) AS interest,
        ROUND(AVG(interest) OVER (PARTITION BY file, term ORDER BY date
                                  ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 1) AS rolling_3m,
        ROUND(100.0 * (interest - LAG(interest, 12) OVER (PARTITION BY file, term ORDER BY date))
              / NULLIF(LAG(interest, 12) OVER (PARTITION BY file, term ORDER BY date), 0), 1) AS yoy_pct
    FROM monthly_all
    ORDER BY file, term, date
''').df()

print(f"Last 12 months for the most searched term ({top}):")
display(monthly_sql[monthly_sql["term"] == top].tail(12))

In [ ]:
yearly_sql = con.sql('''
    SELECT
        file, term, YEAR(date) AS year,
        ROUND(AVG(interest), 1) AS avg_interest,
        ROUND(MAX(interest), 1) AS peak_month_interest,
        ROUND(100.0 * AVG(interest) / SUM(AVG(interest)) OVER (PARTITION BY file, YEAR(date)), 1) AS share_of_file_pct
    FROM monthly_all
    GROUP BY file, term, YEAR(date)
    ORDER BY file, year, avg_interest DESC
''').df()
display(yearly_sql)

**Careful:** 2021 and 2026 are partial years, and year-over-year change on low-volume terms is noisy. Use it for the big terms only.

## 8. The GLP-1 lens (optional, needs a third file)
Hypothesis from industry reports: weight-loss drugs may push people toward scent as a source of pleasure. This section checks whether Dutch interest in GLP-1 drugs moves with interest in gourmand scents. It can show whether the idea is consistent with the data. It cannot prove it.

To use it, download a Google Trends file for **wegovy, mounjaro, ozempic** (Netherlands, past 5 years) and save it as `data/trends_NL_glp1_terms_5y.csv`.

In [ ]:
GLP1_FILE = DATA_DIR / "trends_NL_glp1_terms_5y.csv"
first_file = list(data)[0]

def clean_spike(s):
    s = s.copy()
    if SPIKE_WINDOW:
        s.loc[SPIKE_WINDOW[0]:SPIKE_WINDOW[1]] = np.nan
        s = s.interpolate()
    return s

rescale = lambda s: 100 * s / s.max()
deseason = lambda s: s - s.groupby(s.index.month).transform("mean")

glp_vs_gourmand = None
corr_table = None

if GLP1_FILE.exists():
    glp = drop_partial(load_trends(GLP1_FILE))
    glp_idx = rescale(to_monthly(glp.sum(axis=1)))
    gourmand_idx = rescale(to_monthly(clean_spike(data[first_file].sum(axis=1))))

    glp_vs_gourmand = (pd.concat([gourmand_idx.rename("gourmand_index"),
                                  glp_idx.rename("glp1_index")], axis=1)
                         .dropna().reset_index().rename(columns={"index": "date"}))

    fig, ax = plt.subplots()
    ax.plot(gourmand_idx.index, gourmand_idx, label=f"Gourmand scent terms ({first_file})", linewidth=2)
    ax.plot(glp_idx.index, glp_idx, label="GLP-1 terms (wegovy, mounjaro, ozempic)", linewidth=2)
    ax.set_title("Search interest in the Netherlands, each rescaled to its own peak = 100")
    ax.legend(frameon=False)
    fig.tight_layout()
    fig.savefig(OUT_DIR / "04_glp1_vs_gourmand.png", dpi=150)
    plt.show()

    # Remove each series' normal yearly pattern, then check if they move together.
    g, p = deseason(gourmand_idx), deseason(glp_idx)
    rows = []
    for lag in range(0, 4):
        lv = pd.concat([p.shift(lag), g], axis=1).dropna()
        ch = pd.concat([p.diff().shift(lag), g.diff()], axis=1).dropna()
        rows.append({"GLP-1 leads by (months)": lag,
                     "correlation of levels": round(lv.corr().iloc[0, 1], 2),
                     "correlation of monthly changes": round(ch.corr().iloc[0, 1], 2),
                     "months compared": len(ch)})
    corr_table = pd.DataFrame(rows)
    display(corr_table)
else:
    print("GLP-1 file not found, skipping this section. (That is fine for a first run.)")

**How to read it:** levels look more correlated whenever both lines simply trend upward, so trust the *monthly changes* column more. A clearly positive correlation would be *consistent with* the GLP-1 idea. A weak or zero one means the data does not support it. Both are honest findings. Remember that two rising lines can correlate by coincidence, and search interest is not buying.

## 9. Export tidy tables for Power BI

In [ ]:
fc = pred.reset_index().rename(columns={"index": "date", "mean": "interest",
                                         "pi_lower": "lower", "pi_upper": "upper"})
fc = fc[["date", "interest", "lower", "upper"]].copy()
fc["term"] = top
fc["file"] = first_file
fc["family"] = family_of(top)
fc["language"] = "Dutch" if "Dutch" in first_file else "English"
fc["type"] = "Forecast"

export = pd.concat([monthly_all, fc], ignore_index=True)
export = export[["date", "type", "file", "language", "family", "term", "interest", "lower", "upper"]]
export.to_csv(OUT_DIR / "powerbi_trends_monthly.csv", index=False)
written = ["powerbi_trends_monthly.csv"]

if glp_vs_gourmand is not None:
    glp_vs_gourmand.to_csv(OUT_DIR / "powerbi_glp1_vs_gourmand.csv", index=False)
    corr_table.to_csv(OUT_DIR / "powerbi_glp1_correlation.csv", index=False)
    written += ["powerbi_glp1_vs_gourmand.csv", "powerbi_glp1_correlation.csv"]

print("Written to the output folder:", written)

## 10. Your write-up: the "so what"
Fill these in with plain sentences. This is the part a hiring manager reads first.

1. **What I found:** (2 or 3 findings, with numbers from the cells above)
2. **What it means:** (what this says about gourmand demand in the Netherlands, and how sure I am)
3. **What I would do:** (one recommendation for a brand or a fragrance supplier)
4. **What I could not tell:** (e.g. the cause of the September 2026 spike, whether search interest turns into sales)
5. **Next step:** (add a product-level dataset of notes and ratings to test whether gourmand notes rate and sell well)